In [ ]:
import pandas as pd

# File paths
input_file = 'test_tag_columns.csv'
output_file = 'transformed_text_age.csv'

# 1. Read the CSV file
df = pd.read_csv(input_file)

# 2. Select 'text' and 'Age' columns
filtered_df = df[['text', 'Age']].copy()

# 3. Save the filtered dataset
filtered_df.to_csv(output_file, index=False, encoding='utf-8-sig')

# 4. Preview the results and check entity availability
print("--- Data Filtered Successfully ---")
print(filtered_df.head())
print(f"\nSaved to: {output_file}")
print(f"Total Rows: {len(filtered_df)}")
print(f"Rows with Age entities: {filtered_df['Age'].notna().sum()} | Rows with missing Age: {filtered_df['Age'].isna().sum()}")

--- Data Filtered Successfully ---
                                                text  Age
0  আমি < NAME > । আমার বয়স 27 বছর । সাম্প্রতিক স...   27
1  আপনার প্রশ্নের জন্য ধন্যবাদ । দুশ্চিন্তা কমান ...  NaN
2  Thank you for your question . Your serum Trigl...  NaN
3  Proshno korar jonno dhonnobad . Ei boyosher ba...  NaN
4  বেশ কয়েকদিন যাবত গলায় খুব ব্যাথা সেই সাথে কাশি...  NaN

Saved to: transformed_text_age.csv
Total Rows: 3179
Rows with Age entities: 532 | Rows with missing Age: 2647


In [ ]:
!pip install -q openai tqdm

In [ ]:
import time
import pandas as pd
from tqdm import tqdm
from google.colab import userdata
from openai import OpenAI

# 1. Fetch API key from Colab Secrets
try:
    api_key = userdata.get('OPENROUTER_API_KEY')
except Exception as e:
    raise ValueError("Key 'OPENROUTER_API_KEY' not found in Colab Secrets.") from e

# 2. Initialize OpenAI client configured for OpenRouter
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=api_key
)

# 3. Model Name restored to Luna
MODEL_NAME = "gpt-5.6-luna"

# Prompt Template for Age Replacement
PROMPT_TEMPLATE = """You are given a Bangla medical sentence containing one or more age entities.

Your task is to create a modified version of the sentence by replacing exactly ONE age entity with a different but contextually plausible age.

Rules:
1. Identify the specified age entity in the sentence.
2. Replace exactly ONE occurrence of that age with another age.
3. The replacement must be a different age from the original age.
4. The replacement must NOT be another age already present in the original sentence.
5. The replacement should be medically and contextually plausible in the surrounding sentence context.
6. The replacement should fit naturally into the surrounding sentence without making the sentence medically or linguistically implausible.
7. Maintain unit consistency where appropriate (e.g., replace years with years, months with months).
8. Do not add any additional age entity.
9. Do not remove, add, or modify any other information in the sentence.
10. Keep the ENTIRE sentence structure intact. Do NOT truncate, cut short, or summarize any part of the original text.
11. Do not modify the original NER annotation.
12. Output ONLY the complete modified Bangla sentence from start to finish.
13. Do not provide explanations or identify the replacement.

Example:

Original sentence:
রোগীর বয়স ৩৫ বছর এবং তিনি উচ্চ রক্তচাপে ভুগছেন।

Age entity:
৩৫ বছর

Output:
রোগীর বয়স ৪৫ বছর এবংতিনি উচ্চ রক্তচাপে ভুগছেন।

Now perform the replacement.

Original sentence:
{SENTENCE}

Age entity:
{AGE_ENTITY}

Modified sentence:"""

# 4. Load dataset
input_file = "transformed_text_age.csv"
output_file = "transformed_with_luna_modified_age.csv"

df = pd.read_csv(input_file)

# 5. Filter out empty Age rows before looping so tqdm tracks 100% active API calls
valid_df = df.dropna(subset=['Age']).copy()
print(f"Total dataset size: {len(df)} rows.")
print(f"Found {len(valid_df)} rows with valid Age entities. Generating with '{MODEL_NAME}'...\n")

def get_modified_sentence(sentence, age_entity):
    prompt = PROMPT_TEMPLATE.format(
        SENTENCE=str(sentence).strip(),
        AGE_ENTITY=str(age_entity).strip()
    )

    try:
        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.3
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        print(f"\n[API Error] Entity '{age_entity}': {e}")
        return None

# 6. Run generation on valid rows
modified_sentences = []

for idx, row in tqdm(valid_df.iterrows(), total=len(valid_df)):
    modified_text = get_modified_sentence(row['text'], row['Age'])
    modified_sentences.append(modified_text)
    time.sleep(0.1)

valid_df['modified_text'] = modified_sentences

# 7. Export generated dataset
valid_df.to_csv(output_file, index=False, encoding='utf-8-sig')
print(f"\nGeneration complete! Saved {len(valid_df)} rows to '{output_file}'.")

Total dataset size: 3179 rows.
Found 532 rows with valid Age entities. Generating with 'gpt-5.6-luna'...



100%|██████████| 532/532 [20:49<00:00,  2.35s/it]


Generation complete! Saved 532 rows to 'transformed_with_luna_modified_age.csv'.


In [ ]:
import pandas as pd

# Load final output
df_final = pd.read_csv("transformed_with_luna_modified_age.csv")

# Drop any failed generations
df_clean = df_final.dropna(subset=['Age', 'modified_text']).copy()

clean_output_file = "transformed_with_luna_modified_cleaned_age.csv"
df_clean.to_csv(clean_output_file, index=False, encoding='utf-8-sig')

print(f"Successfully cleaned: {len(df_clean)} rows saved to '{clean_output_file}'.")

Successfully cleaned: 532 rows saved to 'transformed_with_luna_modified_cleaned_age.csv'.
